# Réseaux de neurones artificiels

Notebook d'appui à la rédaction de la section « Méthodes » : principes, avantages et limites
des réseaux de neurones pour la prédiction probabiliste :

1. **Neurone artificiel** et fonctions d'activation ;
2. **Perceptron multicouche** (MLP) : architecture, entraînement, surapprentissage ;
3. **Réseaux récurrents** (RNN) et problème du gradient évanescent ;
4. **LSTM** (et GRU) ;
5. **Architectures temporelles** pour la prédiction d'un match (séquences d'équipes, TCN,
   attention).

L'objectif est de **décrire** les méthodes. Les schémas d'architecture sont dessinés avec
matplotlib ; les illustrations utilisent les vraies données de la LNH
(`nhl_forecast.features.jeu_base`), sans recherche de la meilleure configuration : la
comparaison rigoureuse des modèles est faite dans `03_experimentation/`. Le MLP est entraîné
avec `scikit-learn` (PyTorch n'est pas requis pour ce notebook).

Avec `SAUVER = True`, les figures sont enregistrées en PNG (300 dpi) dans
`reports/map6009/figures/` et les tableaux en CSV dans `reports/map6009/tables/`
(préfixe `methodes_`).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch, Patch
from scipy.special import expit
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import log_loss
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from nhl_forecast.config import PROJECT_ROOT
from nhl_forecast.features.jeu_base import charger_jeu_base
from nhl_forecast.visualization.style import COULEURS, ENCRE, ENCRE_2, appliquer_style

In [ ]:
SAUVER = True  # True : enregistre aussi les figures (PNG) et tableaux (CSV) pour le rapport
FIG_DIR = PROJECT_ROOT / "reports" / "map6009" / "figures"
TAB_DIR = PROJECT_ROOT / "reports" / "map6009" / "tables"

pd.set_option("display.max_colwidth", 120)
appliquer_style()  # couleurs + traits/marqueurs/hachures : lisible en noir et blanc
rng = np.random.default_rng(6009)

CLAIR = "#d6e4f5"  # remplissage clair (bleu très pâle : quasi blanc une fois imprimé)
GRIS_MOYEN = "#9a9a9a"


def sauver(fig, nom):
    if SAUVER:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"{nom}.png", dpi=300, bbox_inches="tight", facecolor="white")


def sauver_tableau(df, nom):
    if SAUVER:
        TAB_DIR.mkdir(parents=True, exist_ok=True)
        df.to_csv(TAB_DIR / f"{nom}.csv", encoding="utf-8")
    return df


# Petits outils de dessin pour les schémas d'architecture
def canevas(largeur, hauteur, xlim, ylim):
    fig, ax = plt.subplots(figsize=(largeur, hauteur))
    ax.set(xlim=xlim, ylim=ylim, aspect="equal")
    ax.axis("off")
    return fig, ax


def noeud(ax, x, y, texte="", r=0.3, fc="white", hachure=None, fs=9, couleur_texte=ENCRE):
    ax.add_patch(Circle((x, y), r, fc=fc, ec=ENCRE, lw=1, hatch=hachure, zorder=3))
    if texte:
        ax.text(x, y, texte, ha="center", va="center", fontsize=fs, color=couleur_texte, zorder=4)


def boite(
    ax,
    x,
    y,
    larg,
    h,
    texte="",
    fc="white",
    hachure=None,
    fs=9,
    couleur_texte=ENCRE,
    lw=1,
    style="round,pad=0.04",
):
    """Boîte centrée en (x, y)."""
    ax.add_patch(
        FancyBboxPatch(
            (x - larg / 2, y - h / 2),
            larg,
            h,
            boxstyle=style,
            fc=fc,
            ec=ENCRE,
            lw=lw,
            hatch=hachure,
            zorder=3,
        )
    )
    if texte:
        ax.text(
            x,
            y,
            texte,
            ha="center",
            va="center",
            fontsize=fs,
            color=couleur_texte,
            zorder=4,
            bbox=dict(fc=fc, ec="none", pad=1.5) if hachure else None,
        )


def fleche(ax, p0, p1, couleur=ENCRE, lw=1.0, style="-|>", ls="-", connexion="arc3", z=2):
    ax.add_patch(
        FancyArrowPatch(
            p0,
            p1,
            arrowstyle=style,
            mutation_scale=10,
            color=couleur,
            lw=lw,
            linestyle=ls,
            connectionstyle=connexion,
            shrinkA=0,
            shrinkB=0,
            zorder=z,
        )
    )


def trait(ax, xs, ys, couleur=ENCRE, lw=1.0, ls="-", z=2):
    ax.plot(xs, ys, color=couleur, lw=lw, ls=ls, marker="", zorder=z)

---
## 1. Neurone artificiel

Un neurone calcule une somme pondérée de ses entrées, ajoute un biais et applique une
**fonction d'activation** non linéaire $\varphi$ :

$$a = \varphi\Big(\sum_{j=1}^{p} w_j x_j + b\Big) = \varphi(\mathbf w^\top \mathbf x + b).$$

Avec $\varphi = \sigma$ (sigmoïde) et un seul neurone, on retrouve exactement la **régression
logistique** : les réseaux de neurones en sont une généralisation.

In [ ]:
fig, ax = canevas(7.5, 3.4, (-0.6, 9.6), (-0.4, 4.4))
entrees = [("$x_1$", 3.6), ("$x_2$", 2.6), ("$\\vdots$", 1.6), ("$x_p$", 0.6)]
for i, (lab, y) in enumerate(entrees):
    if lab == "$\\vdots$":
        ax.text(0.4, y, lab, ha="center", va="center", fontsize=12)
        continue
    noeud(ax, 0.4, y, lab)
    fleche(ax, (0.7, y), (3.65, 2.1 + (y - 2.1) * 0.15))
    w = {0: "$w_1$", 1: "$w_2$", 3: "$w_p$"}[i]
    ax.text(1.9, y - (y - 2.1) * 0.42 + 0.17, w, fontsize=8, ha="center", color=ENCRE_2)
noeud(ax, 4.1, 2.1, "$\\Sigma$", r=0.45, fc=CLAIR, fs=13)
ax.text(4.1, 3.0, "$+\\,b$", ha="center", fontsize=9)
fleche(ax, (4.1, 2.85), (4.1, 2.58))
fleche(ax, (4.55, 2.1), (5.6, 2.1))
ax.text(5.07, 2.3, "$z$", ha="center", fontsize=9)
boite(ax, 6.4, 2.1, 1.5, 0.9, "$\\varphi(z)$", fc="white", hachure="///", fs=11)
fleche(ax, (7.18, 2.1), (8.3, 2.1))
noeud(ax, 8.75, 2.1, "$a$", r=0.4, fc=ENCRE_2, couleur_texte="white", fs=11)
ax.text(0.4, -0.25, "Entrées", ha="center", fontsize=8, color=ENCRE_2)
ax.text(4.1, 1.3, "Somme pondérée", ha="center", fontsize=8, color=ENCRE_2)
ax.text(6.4, 1.3, "Activation", ha="center", fontsize=8, color=ENCRE_2)
ax.text(8.75, 1.4, "Sortie", ha="center", fontsize=8, color=ENCRE_2)
sauver(fig, "methodes_rn_neurone")
plt.show()

**Fonctions d'activation.** La sigmoïde et la tangente hyperbolique saturent pour de grandes
valeurs de $|z|$ (dérivée proche de 0), ce qui ralentit l'apprentissage des réseaux profonds ;
ReLU, $\max(0, z)$, ne sature pas pour $z > 0$ et est le choix par défaut des couches cachées.
La **sortie** d'un modèle de probabilité de victoire utilise une sigmoïde (ou un *softmax*
pour trois issues : victoire en temps réglementaire, en prolongation, défaite).

In [ ]:
z = np.linspace(-4, 4, 400)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(z, expit(z), markevery=0.1, label="Sigmoïde $\\sigma(z)$")
ax.plot(z, np.tanh(z), markevery=0.1, label="Tangente hyperbolique $\\tanh(z)$")
ax.plot(z, np.maximum(0, z), markevery=0.1, label="ReLU $\\max(0, z)$")
ax.axhline(0, color=ENCRE_2, lw=0.8)
ax.axvline(0, color=ENCRE_2, lw=0.8)
ax.set(xlabel="$z$", ylabel="$\\varphi(z)$", ylim=(-1.2, 2.2))
ax.legend(fontsize=8, loc="upper left")
sauver(fig, "methodes_rn_activations")
plt.show()

---
## 2. Perceptron multicouche (MLP)

Un MLP empile des **couches** de neurones entièrement connectées. Pour $L$ couches cachées :

$$\mathbf h^{(1)} = \varphi(\mathbf W^{(1)} \mathbf x + \mathbf b^{(1)}),\quad \dots,\quad
\mathbf h^{(L)} = \varphi(\mathbf W^{(L)} \mathbf h^{(L-1)} + \mathbf b^{(L)}),\quad
\hat p = \sigma(\mathbf w^\top \mathbf h^{(L)} + b).$$

Les poids sont appris par **rétropropagation du gradient** en minimisant la log loss (entropie
croisée binaire) avec une variante de la descente de gradient stochastique (ex. Adam). Le
théorème d'approximation universelle garantit qu'un MLP suffisamment large peut approcher
toute fonction continue : la difficulté est de **généraliser** avec peu de données.

Exemple d'architecture pour un match : les variables de l'équipe locale et de l'équipe
visiteuse (et du contexte du match) sont concaténées en entrée.

In [ ]:
fig, ax = canevas(9, 4.8, (-1.8, 10.2), (-0.9, 6.0))
couches = [
    (0.6, [5.2, 4.5, 3.8, 2.6, 1.9, 1.2, 0.2], "Entrée\n$\\mathbf{x}$"),
    (3.4, [4.6, 3.7, 2.8, 1.9, 1.0], "Couche cachée 1\n(ReLU)"),
    (6.0, [4.2, 3.2, 2.2, 1.2], "Couche cachée 2\n(ReLU)"),
    (8.6, [2.7], "Sortie\n(sigmoïde)"),
]
for (x0, ys0, _), (x1, ys1, _) in zip(couches[:-1], couches[1:], strict=True):
    for y0 in ys0:
        for y1 in ys1:
            trait(ax, [x0, x1], [y0, y1], couleur=GRIS_MOYEN, lw=0.5, z=1)
styles_entree = [("white", None)] * 3 + [("white", "////")] * 3 + [(CLAIR, "...")]
for (x, ys, titre), k in zip(couches, range(4), strict=True):
    for i, y in enumerate(ys):
        if k == 0:
            fc, h = styles_entree[i]
            noeud(ax, x, y, r=0.24, fc=fc, hachure=h)
        elif k == 3:
            noeud(ax, x, y, "$\\hat p$", r=0.36, fc=ENCRE_2, couleur_texte="white")
        else:
            noeud(ax, x, y, r=0.24, fc=CLAIR)
    ax.text(x, -0.6, titre, ha="center", va="center", fontsize=8.5)
for y0, y1, lab in [
    (3.75, 5.45, "Équipe\nlocale"),
    (1.15, 2.85, "Équipe\nvisiteuse"),
    (-0.05, 0.45, "Contexte"),
]:
    trait(ax, [-0.05, -0.15, -0.15, -0.05], [y0, y0, y1, y1], lw=0.8)
    ax.text(-0.3, (y0 + y1) / 2, lab, ha="right", va="center", fontsize=8)
ax.annotate(
    "$P(\\text{victoire locale})$",
    xy=(8.96, 2.7),
    xytext=(9.1, 3.6),
    fontsize=8.5,
    ha="left",
    arrowprops=dict(arrowstyle="-", color=ENCRE_2, lw=0.7),
)
sauver(fig, "methodes_rn_mlp_architecture")
plt.show()

**Surapprentissage.** Un MLP a beaucoup de paramètres relativement au nombre de matchs
disponibles (≈ 1 300 par saison). Sans contrôle, la log loss d'entraînement continue de
diminuer alors que celle de validation remonte. Les remèdes classiques : **arrêt précoce**
(on conserve les poids au minimum de la perte de validation), pénalité $L_2$ sur les poids
(*weight decay*), *dropout*, réseau plus petit, normalisation des entrées.

Illustration sur les matchs de la LNH : 13 variables d'écart local − visiteur (mêmes variables
que dans `02_apprentissage_automatique`), entraînement sur trois saisons (2018-2019 à
2020-2021, ordre de grandeur des données d'un modèle récent), validation sur 2021-2022. MLP de
deux couches cachées de 128 neurones, avec et sans pénalité $L_2$.

In [ ]:
VARS = [
    "d_pts_pct_saison", "d_diff_buts_saison", "d_tirs_pct_saison", "d_pp_pct_saison",
    "d_pk_pct_saison", "d_xg_pct_saison", "d_corsi_pct_saison", "d_xg_pour_saison",
    "d_xg_contre_saison", "d_pts_pct_10", "d_xg_pct_10", "d_repos", "d_partant_sv_saison",
]  # fmt: skip
matchs, equipes = charger_jeu_base()
jeu = matchs.dropna(subset=VARS)
tr_m, va_m = jeu.saison.between(2018, 2020), jeu.saison == 2021
sc = StandardScaler().fit(jeu.loc[tr_m, VARS])
X_tr, y_tr = sc.transform(jeu.loc[tr_m, VARS]), jeu.loc[tr_m, "victoire_locale"].to_numpy()
X_va, y_va = sc.transform(jeu.loc[va_m, VARS]), jeu.loc[va_m, "victoire_locale"].to_numpy()
LL_CONST = log_loss(y_va, np.full(len(y_va), y_tr.mean()))
print(
    f"Entraînement : {len(y_tr):,} matchs ; validation : {len(y_va):,} matchs ; "
    f"log loss constante (validation) : {LL_CONST:.4f}"
)

N_EPOQUES = 100
ALPHA = 1.0
courbes = {}
for alpha in [0.0, ALPHA]:
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 128),
        alpha=alpha,
        learning_rate_init=3e-4,
        batch_size=64,
        random_state=0,
    )
    tr, va = [], []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        for _ in range(N_EPOQUES):
            mlp.partial_fit(X_tr, y_tr, classes=[0, 1])
            tr.append(log_loss(y_tr, mlp.predict_proba(X_tr)[:, 1]))
            va.append(log_loss(y_va, mlp.predict_proba(X_va)[:, 1]))
    courbes[alpha] = (np.array(tr), np.array(va))

ep = np.arange(1, N_EPOQUES + 1)
fig, ax = plt.subplots(figsize=(7, 3.4))
for alpha, c, mk in [(0.0, COULEURS[1], "s"), (ALPHA, COULEURS[0], "o")]:
    tr, va = courbes[alpha]
    lab = "sans pénalité" if alpha == 0 else f"pénalité $L_2$ ($\\alpha = {alpha:g}$)"
    ax.plot(ep, va, color=c, ls="-", marker=mk, markevery=15, label=f"Validation, {lab}")
    ax.plot(
        ep, tr, color=c, ls="--", marker=mk, markevery=15, mfc="white", label=f"Entraînement, {lab}"
    )
tr0, va0 = courbes[0.0]
e_min = int(np.argmin(va0))
ax.axvline(ep[e_min], color=ENCRE_2, ls=":", lw=1)
ax.annotate(
    "Arrêt précoce",
    xy=(ep[e_min], va0[e_min]),
    xytext=(ep[e_min] + 12, 0.75),
    fontsize=8,
    arrowprops=dict(arrowstyle="->", color=ENCRE_2, lw=0.8),
)
ax.axhline(
    LL_CONST, color=ENCRE, ls=":", marker="", lw=1, label="Probabilité constante (validation)"
)
ax.set(xlabel="Époque", ylabel="Log loss", ylim=(0.45, 0.8))
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
sauver(fig, "methodes_rn_courbes_apprentissage")
plt.show()
print(
    f"Log loss de validation minimale sans pénalité : {va0.min():.4f} (époque {ep[e_min]}) ; "
    f"finale : {va0[-1]:.4f} ; avec pénalité, minimale : {courbes[ALPHA][1].min():.4f}, "
    f"finale : {courbes[ALPHA][1][-1]:.4f}"
)

**Avantages (MLP).** Approximateur universel : capte non-linéarités et interactions entre
toutes les variables ; entrées de nature variée (variables continues, plongements d'équipes et
de joueurs) ; optimise directement la log loss ; brique de base des architectures plus
complexes.

**Limites (MLP).** Gourmand en données ; nombreux hyperparamètres (architecture, taux
d'apprentissage, régularisation, époques) ; résultats variables selon l'initialisation ;
probabilités parfois **surconfiantes** (calibration à vérifier, ex. *temperature scaling*) ;
peu interprétable ; sur données tabulaires de taille modeste, rarement meilleur que le gradient
boosting.

---
## 3. Réseaux de neurones récurrents (RNN)

Un RNN traite une **séquence** $\mathbf x_1, \dots, \mathbf x_T$ (ex. les $T$ derniers matchs
d'une équipe) en mettant à jour un **état caché** qui résume l'historique :

$$\mathbf h_t = \tanh(\mathbf W_h \mathbf h_{t-1} + \mathbf W_x \mathbf x_t + \mathbf b).$$

Les **mêmes poids** sont réutilisés à chaque pas de temps. Le dernier état $\mathbf h_T$ sert
d'entrée à une couche de sortie. L'entraînement se fait par rétropropagation à travers le temps
(*BPTT*), sur le réseau « déplié ».

In [ ]:
fig, ax = canevas(9.5, 3.6, (-0.6, 13.4), (-0.6, 4.6))
# Forme compacte avec boucle
boite(ax, 1.0, 2.0, 1.3, 0.9, "RNN", fc=CLAIR)
noeud(ax, 1.0, 0.2, "$\\mathbf{x}_t$", r=0.38, fs=9)
fleche(ax, (1.0, 0.58), (1.0, 1.5))
fleche(ax, (1.0, 2.5), (1.0, 3.45))
noeud(ax, 1.0, 3.85, "$\\mathbf{h}_t$", r=0.38, fs=9)
fleche(ax, (1.68, 2.25), (1.68, 1.75), connexion="arc3,rad=-2.2")
ax.text(2.3, 2.0, "$\\mathbf{W}_h$", fontsize=8, va="center")
ax.text(3.0, 2.0, "$=$", fontsize=16, va="center", ha="center")
# Forme dépliée
etapes = [(4.6, "t-2"), (7.0, "t-1"), (9.4, "t")]
for i, (x, t) in enumerate(etapes):
    boite(ax, x, 2.0, 1.3, 0.9, "RNN", fc=CLAIR)
    noeud(ax, x, 0.2, f"$\\mathbf{{x}}_{{{t}}}$", r=0.42, fs=8.5)
    fleche(ax, (x, 0.62), (x, 1.5))
    noeud(ax, x, 3.85, f"$\\mathbf{{h}}_{{{t}}}$", r=0.42, fs=8.5)
    fleche(ax, (x, 2.5), (x, 3.43))
    fleche(ax, (x + 0.7, 2.0), (x + 1.7, 2.0) if i < 2 else (x + 2.2, 2.0))
    ax.text(
        x + 1.2,
        2.15,
        "$\\mathbf{W}_h$" if i < 2 else "$\\mathbf{h}_t$",
        fontsize=7.5,
        ha="center",
        va="bottom",
    )
ax.text(3.55, 2.0, "$\\cdots$", fontsize=12, va="center", ha="center")
fleche(ax, (3.75, 2.0), (3.93, 2.0))
boite(ax, 12.3, 2.0, 1.4, 0.9, "$\\sigma$\nsortie", fc=ENCRE_2, couleur_texte="white", fs=8.5)
ax.text(12.3, 3.0, "$\\hat p$", ha="center", fontsize=10)
fleche(ax, (12.3, 2.5), (12.3, 2.8))
for x, t in etapes:
    ax.text(x, -0.55, f"Match {t.replace('-', ' − ')}", ha="center", fontsize=7.5, color=ENCRE_2)
sauver(fig, "methodes_rn_rnn_deplie")
plt.show()

**Gradient évanescent.** Le gradient de la perte par rapport à un état ancien est un produit de
Jacobiens : $\partial \mathbf h_T / \partial \mathbf h_t = \prod_{k=t+1}^{T}
\partial \mathbf h_k / \partial \mathbf h_{k-1}$. Dans un RNN simple, chaque facteur vaut
environ $w \cdot \tanh'(\cdot) < 1$ et le produit **décroît exponentiellement** : le réseau
n'apprend pas les dépendances lointaines (ou explose si les facteurs sont $> 1$). Dans un LSTM,
le chemin de la cellule mémoire fait intervenir le produit des **portes d'oubli** $f_t$, que le
réseau peut maintenir proches de 1.

Illustration schématique (simulation de 2 000 séquences ; médiane du produit des facteurs) :

In [ ]:
T, N_SIM = 60, 2_000
k = np.arange(T + 1)


def produit_median(facteurs):
    return np.median(np.c_[np.ones(N_SIM), np.cumprod(facteurs, axis=1)], axis=0)


a = rng.normal(0, 1, (N_SIM, T))
series = [
    ("RNN simple, $w = 1$", produit_median(1.0 * (1 - np.tanh(a) ** 2))),
    ("RNN simple, $w = 1{,}5$", produit_median(1.5 * (1 - np.tanh(a) ** 2))),
    (
        "LSTM, porte d'oubli $f_t \\approx 0{,}95$",
        produit_median(expit(rng.normal(3.0, 0.5, (N_SIM, T)))),
    ),
    (
        "LSTM, porte d'oubli $f_t \\approx 0{,}99$",
        produit_median(expit(rng.normal(4.6, 0.5, (N_SIM, T)))),
    ),
]
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for nom, s in series:
    ax.plot(k, s, markevery=6, label=nom)
ax.set_yscale("log")
ax.set_ylim(1e-12, 2)
ax.set_xlabel("Distance temporelle $T - t$ (nombre de matchs)")
ax.set_ylabel("Facteur de gradient (médiane)")
ax.legend(fontsize=8, loc="lower left")
sauver(fig, "methodes_rn_gradient_evanescent")
plt.show()

---
## 4. LSTM (*Long Short-Term Memory*)

Le LSTM (Hochreiter et Schmidhuber, 1997) ajoute une **cellule mémoire** $\mathbf c_t$ et trois
**portes** (valeurs entre 0 et 1) qui contrôlent le flux d'information. Avec
$[\mathbf h_{t-1}, \mathbf x_t]$ la concaténation de l'état précédent et de l'entrée :

$$\begin{aligned}
\mathbf f_t &= \sigma(\mathbf W_f [\mathbf h_{t-1}, \mathbf x_t] + \mathbf b_f) &&\text{porte d'oubli}\\
\mathbf i_t &= \sigma(\mathbf W_i [\mathbf h_{t-1}, \mathbf x_t] + \mathbf b_i) &&\text{porte d'entrée}\\
\tilde{\mathbf c}_t &= \tanh(\mathbf W_c [\mathbf h_{t-1}, \mathbf x_t] + \mathbf b_c) &&\text{candidat}\\
\mathbf c_t &= \mathbf f_t \odot \mathbf c_{t-1} + \mathbf i_t \odot \tilde{\mathbf c}_t &&\text{mise à jour de la mémoire}\\
\mathbf o_t &= \sigma(\mathbf W_o [\mathbf h_{t-1}, \mathbf x_t] + \mathbf b_o) &&\text{porte de sortie}\\
\mathbf h_t &= \mathbf o_t \odot \tanh(\mathbf c_t) &&\text{état caché}
\end{aligned}$$

La mise à jour **additive** de $\mathbf c_t$ crée un chemin où le gradient circule sans être
écrasé. Le **GRU** (Cho et al., 2014) est une variante simplifiée à deux portes, souvent aussi
performante avec moins de paramètres.

Dans le contexte du hockey, la porte d'oubli permet en principe d'apprendre combien de temps la
« forme » d'une équipe persiste, au lieu de fixer à la main une fenêtre ou un facteur de
lissage exponentiel.

In [ ]:
fig, ax = canevas(9, 5.4, (-0.9, 12.9), (-0.9, 7.2))
# Cadre de la cellule
ax.add_patch(
    FancyBboxPatch(
        (1.3, 0.5), 9.4, 5.6, boxstyle="round,pad=0.1", fc="#f4f4f2", ec=ENCRE_2, lw=1, zorder=0
    )
)
Y_C, Y_H, Y_PORTES = 5.3, 1.0, 2.6
# Chemin de la cellule mémoire (haut)
trait(ax, [-0.4, 11.7], [Y_C, Y_C], lw=2.2)
fleche(ax, (11.4, Y_C), (12.2, Y_C), lw=2.2)
ax.text(-0.5, Y_C + 0.35, "$\\mathbf{c}_{t-1}$", fontsize=10, ha="left")
ax.text(12.3, Y_C, "$\\mathbf{c}_t$", fontsize=10, va="center")
# Entrées en bas : h_{t-1} et x_t concaténés (bus)
trait(ax, [-0.4, 7.6], [Y_H, Y_H], lw=1.4)
ax.text(-0.5, Y_H - 0.4, "$\\mathbf{h}_{t-1}$", fontsize=10, ha="left")
trait(ax, [2.0, 2.0], [-0.6, Y_H], lw=1.4)
ax.add_patch(Circle((2.0, Y_H), 0.07, color=ENCRE, zorder=3))
ax.text(2.0, -0.85, "$\\mathbf{x}_t$", fontsize=10, ha="center")
# Portes : position x, nom, activation
portes = [
    (3.0, "$\\sigma$", "$\\mathbf{f}_t$"),
    (4.6, "$\\sigma$", "$\\mathbf{i}_t$"),
    (6.1, "tanh", "$\\tilde{\\mathbf{c}}_t$"),
    (7.6, "$\\sigma$", "$\\mathbf{o}_t$"),
]
for x, act, nom in portes:
    trait(ax, [x, x], [Y_H, Y_PORTES - 0.35])
    est_tanh = act == "tanh"
    boite(
        ax,
        x,
        Y_PORTES,
        0.95,
        0.7,
        act,
        fc="white" if not est_tanh else CLAIR,
        hachure="////" if est_tanh else None,
        fs=10,
    )
    ax.text(x + 0.55, Y_PORTES + 0.55, nom, fontsize=9)
# Porte d'oubli -> × sur le chemin de la cellule
noeud(ax, 3.0, Y_C, "$\\times$", r=0.3, fs=12)
fleche(ax, (3.0, Y_PORTES + 0.38), (3.0, Y_C - 0.3))
# Porte d'entrée × candidat -> + sur le chemin de la cellule
noeud(ax, 5.35, 4.0, "$\\times$", r=0.3, fs=12)
trait(ax, [4.6, 4.6, 5.05], [Y_PORTES + 0.38, 4.0, 4.0])
trait(ax, [6.1, 6.1, 5.65], [Y_PORTES + 0.38, 4.0, 4.0])
noeud(ax, 5.35, Y_C, "$+$", r=0.3, fs=12)
fleche(ax, (5.35, 4.3), (5.35, Y_C - 0.3))
# Sortie : tanh(c_t) × o_t -> h_t
boite(ax, 9.1, 4.2, 1.0, 0.6, "tanh", fc=CLAIR, hachure="////", fs=9, style="round,pad=0.12")
fleche(ax, (9.1, Y_C), (9.1, 4.55))
noeud(ax, 9.1, 2.6, "$\\times$", r=0.3, fs=12)
fleche(ax, (9.1, 3.85), (9.1, 2.9))
fleche(ax, (8.08, Y_PORTES), (8.8, Y_PORTES))

trait(ax, [9.4, 10.2, 10.2], [2.6, 2.6, Y_H])
fleche(ax, (10.2, Y_H), (12.2, Y_H), lw=1.4)
ax.text(12.3, Y_H, "$\\mathbf{h}_t$", fontsize=10, va="center")
trait(ax, [10.2, 10.2], [2.6, 6.4])
fleche(ax, (10.2, 6.3), (10.2, 7.0))
ax.add_patch(Circle((10.2, 2.6), 0.07, color=ENCRE, zorder=3))
ax.text(
    10.4, 6.9, "$\\mathbf{h}_t$ (vers la sortie\nou la couche suivante)", fontsize=8, va="center"
)
# Légende
ax.legend(
    handles=[
        Patch(fc="white", ec=ENCRE, label="Couche sigmoïde (porte)"),
        Patch(fc=CLAIR, ec=ENCRE, hatch="////", label="Couche tanh"),
        plt.Line2D(
            [],
            [],
            marker="o",
            ms=11,
            mfc="white",
            mec=ENCRE,
            ls="",
            label="Opération élément par élément",
        ),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, 0.0),
    ncols=3,
    fontsize=8,
    frameon=False,
)
sauver(fig, "methodes_rn_lstm_cellule")
plt.show()

**Avantages (RNN/LSTM).** Traitent directement des **séquences** de matchs sans résumer la forme
récente par des moyennes mobiles choisies à la main ; mémoire de longue portée (LSTM/GRU) ;
poids partagés dans le temps ; peuvent combiner historique d'équipe et de joueurs.

**Limites (RNN/LSTM).** Encore plus gourmands en données que le MLP ; entraînement séquentiel
lent et instable ; nombreux choix (longueur de séquence, taille de l'état, réinitialisation
entre saisons, traitement des équipes qui changent d'effectif) ; gain sur des statistiques
agrégées pas toujours supérieur à de bonnes variables de forme récente ; interprétation
difficile.

---
## 5. Architectures temporelles pour la prédiction d'un match

**Pourquoi des séquences ?** Les performances d'une équipe d'un match à l'autre sont
corrélées : la corrélation entre la statistique d'un match et celle du $k$-ième match suivant
(même équipe, même saison) est faible, mais elle reste positive même à 40 matchs d'écart. Elle
reflète surtout le **niveau stable** de l'équipe ; la légère décroissance des premiers
décalages correspond à la **forme passagère**. Le résultat (victoire) est beaucoup plus bruité
que les statistiques de jeu comme le Corsi% ou le xG%. Les variables « saison à date » et
« 10 derniers matchs » résument cet historique avec des pondérations fixées à la main ; un
modèle récurrent apprend lui-même quelle partie de l'historique conserver.

In [ ]:
eq = equipes.sort_values(["date", "gameId"]).copy()
eq["victoire"] = eq["wins"].astype(float)
SERIES_AC = {"xg_pct": "xG% du match", "corsi_pct": "Corsi% du match", "victoire": "victoire (0/1)"}
decalages = np.arange(1, 41)
ac = {}
for col in SERIES_AC:
    g = eq.dropna(subset=[col]).groupby(["saison", "teamId"])[col]
    ac[col] = [eq[col].corr(g.shift(-k)) for k in decalages]

fig, ax = plt.subplots(figsize=(6.5, 3.3))
for col, lab in SERIES_AC.items():
    ax.plot(decalages, ac[col], markevery=4, label=lab)
ax.axhline(0, color=ENCRE_2, lw=0.8)
ax.set_xlabel("Décalage $k$ (matchs de la même équipe, même saison)")
ax.set_ylabel("Corrélation entre le match $t$ et $t + k$")
ax.legend(fontsize=8)
sauver(fig, "methodes_rn_autocorrelation")
plt.show()

**Entrée d'un modèle séquentiel.** Pour chaque match à prédire, l'entrée d'un LSTM est une
matrice (pas de temps × variables) : les statistiques des $N$ derniers matchs de chaque équipe,
dans l'ordre chronologique. Exemple : les 10 matchs des Canadiens précédant leur dernier match
de la saison 2024-2025 (niveaux de gris : écart à la moyenne de la ligue, en écarts-types ;
valeurs brutes indiquées).

In [ ]:
STATS_SEQ = {
    "buts_pour": "Buts pour",
    "buts_contre": "Buts contre",
    "tirs_pct": "Part des tirs",
    "xg_pct": "xG%",
    "corsi_pct": "Corsi%",
    "victoire": "Victoire",
}
MTL = int(eq.loc[eq.teamFullName.str.startswith("Montr"), "teamId"].iloc[0])
seq = eq[(eq.teamId == MTL) & (eq.saison == 2024)].iloc[-11:-1]
z = (seq[list(STATS_SEQ)] - eq[list(STATS_SEQ)].mean()) / eq[list(STATS_SEQ)].std()

fig, ax = plt.subplots(figsize=(8, 3.4))
im = ax.imshow(z.T.clip(-2, 2), cmap="Greys", vmin=-2.5, vmax=2.5, aspect="auto")
for i, col in enumerate(STATS_SEQ):
    for j, v in enumerate(seq[col]):
        txt = f"{v:.0f}" if col in ("buts_pour", "buts_contre", "victoire") else f"{100 * v:.0f}"
        ax.text(
            j,
            i,
            txt,
            ha="center",
            va="center",
            fontsize=8,
            color="white" if z.iloc[j, i] > 1 else ENCRE,
        )
ax.set_yticks(
    range(len(STATS_SEQ)),
    [
        f"{lab}" + (" (%)" if c in ("tirs_pct", "xg_pct", "corsi_pct") else "")
        for c, lab in STATS_SEQ.items()
    ],
    fontsize=8,
)
ax.set_xticks(range(10), [f"$t-{10 - j}$" for j in range(10)], fontsize=8)
ax.set_xlabel(
    "Matchs précédents (du plus ancien au plus récent) → "
    "entrée $\\mathbf{x}_{t-10}, \\dots, \\mathbf{x}_{t-1}$"
)
ax.grid(False)
fig.colorbar(im, ax=ax, label="Écart à la moyenne (é.-t.)", shrink=0.9)
sauver(fig, "methodes_rn_sequence_entree")
plt.show()

**Architecture.** Un match oppose **deux** séquences : l'historique récent de l'équipe locale et celui de
l'équipe visiteuse. Architecture typique (« siamoise ») : un **même** encodeur LSTM (poids
partagés) résume chaque séquence en un vecteur, puis les deux résumés sont concaténés avec les
variables propres au match (repos, voyage, gardien partant) avant des couches denses et une
sortie sigmoïde. Seuls les matchs **antérieurs** au match prédit entrent dans les séquences
(aucune fuite d'information).

In [ ]:
fig, ax = canevas(10, 5.4, (-0.6, 15.6), (-0.4, 8.0))
N_SEQ = 4
for y0, nom, hachure in [(6.0, "Équipe locale", None), (1.4, "Équipe visiteuse", "////")]:
    xs = [0.8 + 1.6 * i for i in range(N_SEQ)]
    etiquettes = ["$t-N$", "$\\cdots$", "$t-2$", "$t-1$"]
    for i, (x, lab) in enumerate(zip(xs, etiquettes, strict=True)):
        boite(
            ax,
            x,
            y0 - 1.3,
            1.05,
            0.65,
            lab if lab != "$\\cdots$" else "",
            fc="white",
            hachure=hachure,
            fs=8,
        )
        if lab == "$\\cdots$":
            ax.text(
                x,
                y0 - 1.3,
                "$\\cdots$",
                ha="center",
                va="center",
                fontsize=11,
                zorder=5,
                bbox=dict(fc="white", ec="none", pad=0.5),
            )
        boite(ax, x, y0, 1.05, 0.75, "LSTM", fc=CLAIR, fs=8)
        fleche(ax, (x, y0 - 0.97), (x, y0 - 0.4))
        if i < N_SEQ - 1:
            fleche(ax, (x + 0.55, y0), (x + 1.05, y0))
    ax.text(xs[0] - 0.65, y0 + 0.75, nom, fontsize=9, ha="left", fontweight="bold")
    ax.text(
        np.mean(xs),
        y0 - 2.0,
        "Matchs précédents (variables par match)",
        fontsize=7.5,
        ha="center",
        color=ENCRE_2,
    )
    # Résumé de la séquence
    noeud(ax, 7.6, y0, "$\\mathbf{h}$", r=0.38, fc=CLAIR, fs=9)
    fleche(ax, (xs[-1] + 0.55, y0), (7.22, y0))
    fleche(ax, (7.98, y0), (9.15, 4.15 + (0.5 if y0 > 4 else -0.5)))
ax.text(
    3.2,
    3.0,
    "Même encodeur LSTM (poids partagés)",
    fontsize=8,
    ha="center",
    style="italic",
    color=ENCRE_2,
)
# Variables du match
boite(ax, 7.6, 3.7, 2.1, 0.75, "Contexte du match\n(repos, gardien…)", fc="white", fs=7.5)
fleche(ax, (8.66, 3.7), (9.15, 3.95))
# Concaténation -> denses -> sortie
boite(ax, 9.75, 4.15, 1.1, 2.4, "Concat.", fc="white", fs=8)
fleche(ax, (10.3, 4.15), (10.95, 4.15))
boite(ax, 11.7, 4.15, 1.4, 1.6, "Couches\ndenses", fc=CLAIR, fs=8)
fleche(ax, (12.4, 4.15), (13.05, 4.15))
boite(ax, 13.65, 4.15, 1.1, 0.9, "$\\sigma$", fc=ENCRE_2, couleur_texte="white", fs=11)
fleche(ax, (14.2, 4.15), (14.75, 4.15))
ax.text(14.85, 4.15, "$\\hat p$", fontsize=11, va="center")
ax.text(13.65, 3.35, "$P(\\text{victoire locale})$", fontsize=7.5, ha="center", color=ENCRE_2)
sauver(fig, "methodes_rn_architecture_temporelle")
plt.show()

**Autres architectures temporelles.**

- **Réseaux convolutifs temporels (TCN)** : convolutions **causales** (seul le passé est vu)
  et **dilatées** (écart $d = 1, 2, 4, \dots$ entre les entrées combinées). Le champ récepteur
  croît exponentiellement avec la profondeur, l'entraînement est parallélisable et le gradient
  ne traverse pas une longue chaîne récurrente.
- **Mécanismes d'attention et Transformers** : chaque pas de temps pondère directement les
  matchs passés jugés pertinents ; très performants sur de grands corpus, mais coûteux en données
  pour des séries courtes comme une saison de hockey.

Schéma du champ récepteur d'un TCN à trois couches (taille de noyau 2, dilatations 1, 2, 4) :
la sortie au match $t$ dépend des 8 matchs précédents.

In [ ]:
fig, ax = canevas(9, 3.6, (-1.6, 15.6), (-0.7, 4.0))
n_pas = 16
niveaux = [(0, "Entrée"), (1, "$d = 1$"), (2, "$d = 2$"), (3, "$d = 4$")]
# Nœuds actifs (qui contribuent à la sortie au dernier pas) par niveau
actifs = {3: {15}}
for niv, d in [(3, 4), (2, 2), (1, 1)]:
    actifs[niv - 1] = {t for a in actifs[niv] for t in (a, a - d)}
for niv, d in [(1, 1), (2, 2), (3, 4)]:
    for t in range(n_pas):
        for s in (t, t - d):
            if s < 0:
                continue
            actif = t in actifs[niv] and s in actifs[niv - 1]
            trait(
                ax,
                [s, t],
                [niv - 1, niv],
                couleur=ENCRE if actif else "#d0d0d0",
                lw=1.4 if actif else 0.5,
                z=2 if actif else 1,
            )
for niv, lab in niveaux:
    for t in range(n_pas):
        actif = t in actifs[niv]
        fc = ENCRE_2 if (actif and niv == 3) else (CLAIR if actif else "white")
        ax.add_patch(
            Circle(
                (t, niv),
                0.2,
                fc=fc,
                ec=ENCRE if actif else "#a0a0a0",
                lw=0.9,
                zorder=3,
                hatch="////" if (actif and niv == 0) else None,
            )
        )
    ax.text(-0.7, niv, lab, ha="right", va="center", fontsize=8)
ax.text(15, 3.45, "Sortie au match $t$", ha="center", fontsize=8)
ax.annotate(
    "",
    xy=(15.4, -0.5),
    xytext=(7.6, -0.5),
    arrowprops=dict(arrowstyle="<->", color=ENCRE_2, lw=0.8),
)
ax.text(11.5, -0.62, "Champ récepteur : 8 matchs", ha="center", va="top", fontsize=8, color=ENCRE_2)
ax.text(0, -0.62, "Passé", ha="center", va="top", fontsize=8, color=ENCRE_2)
sauver(fig, "methodes_rn_tcn_dilatee")
plt.show()

---
## 6. Tableau synthèse

In [ ]:
synthese = pd.DataFrame(
    [
        (
            "MLP",
            "Couches denses avec activations non linéaires, rétropropagation",
            "Vecteur de variables (locale, visiteuse, contexte)",
            "Approximateur universel, interactions automatiques, optimise la log loss",
            "Gourmand en données, hyperparamètres nombreux, "
            "surconfiance possible, peu interprétable",
        ),
        (
            "RNN simple",
            "État caché mis à jour à chaque pas de temps, poids partagés",
            "Séquence des derniers matchs d'une équipe",
            "Traite des séquences de longueur variable, modèle la dynamique",
            "Gradient évanescent/explosif, mémoire courte, entraînement instable",
        ),
        (
            "LSTM / GRU",
            "Cellule mémoire et portes (oubli, entrée, sortie)",
            "Séquences de matchs (équipes, joueurs)",
            "Dépendances de longue portée, persistance de la forme apprise",
            "Données et calcul importants, nombreux choix de conception, interprétation difficile",
        ),
        (
            "TCN / attention",
            "Convolutions causales dilatées / pondération des pas passés",
            "Séquences de matchs",
            "Parallélisable, champ récepteur contrôlé (TCN), pondération adaptative (attention)",
            "Très gourmands en données (Transformers), peu utilisés en prédiction LNH",
        ),
    ],
    columns=["Méthode", "Principe", "Données d'entrée", "Avantages", "Limites"],
).set_index("Méthode")
sauver_tableau(synthese, "methodes_tab_reseaux_neurones")
synthese